# AvesCO en Google Colab
Crea el secreto **NVIDIA_API_KEY** en el panel de la llave, habilita el acceso para este notebook y ejecuta las tres celdas en orden. Puedes usar CPU.

In [ ]:
import subprocess
import sys
from pathlib import Path

repositorio = Path("/content/AvesCO")
if not repositorio.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/JulianJ07/RAG_Detector_Ave.git", str(repositorio)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(repositorio / "requirements.txt")], check=True)


In [ ]:
import os
from google.colab import userdata

try:
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY").strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as error:
    raise RuntimeError("Crea NVIDIA_API_KEY en Secretos y habilita el acceso.") from error
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"


Abre el enlace de Gradio que imprime la siguiente celda e inicia sesión con las credenciales que aparecen en la salida. La primera ejecución crea el índice con NVIDIA.

In [ ]:
import secrets
sys.path.insert(0, str(repositorio))

from app import CSS, construir_interfaz
from avesco.asistente import AsistenteAves
from avesco.configuracion import Configuracion
from avesco.datos import Catalogo
from avesco.indice import crear_indice

config = Configuracion.cargar()
catalogo = Catalogo.cargar()
indice = crear_indice(config, catalogo)
demo = None
try:
    demo = construir_interfaz(AsistenteAves(config, catalogo, indice, indice.modelo))
    contrasena = secrets.token_urlsafe(12)
    print("Usuario de Gradio: avesco")
    print(f"Contraseña de Gradio: {contrasena}")
    demo.launch(share=True, inline=False, auth=("avesco", contrasena), css=CSS, show_error=True)
finally:
    try:
        if demo is not None:
            demo.close()
    finally:
        indice.cerrar()
